# ЛР 02.1 — Глобальная интерпретация моделей (solution)

Базовый маршрут: выбрать лучший неполный набор признаков (feature set) из ЛР 01, обучить `LogisticRegression` и `RandomForestClassifier`, затем сохранить глобальные explainability-артефакты.

In [ ]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

prepared = {}
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    feature_set_name = lab.choose_best_nonfull_feature_set(model_results, feature_sets, dataset_name)
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_set_name': feature_set_name,
        'selected_features': feature_sets[dataset_name][feature_set_name],
    }

artifacts = {}
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    artifacts[dataset_name] = {
        'LogisticRegression': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        ),
        'RandomForest': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=3,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    }


In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
importance_frames = []
summary_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        importance_frames.append(
            lab.build_global_importance_table(
                artifact=artifact,
                dataset_name=dataset_name,
                model_name=model_name,
                feature_set_name=feature_set_name,
            )
        )
        _, pd_summary = lab.build_partial_dependence_summary(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=feature_set_name,
        )
        summary_frames.append(pd_summary)

global_importance_comparison = pd.concat(importance_frames, ignore_index=True)
partial_dependence_summary = pd.concat(summary_frames, ignore_index=True)

# Сохраняем таблицу артефакта в CSV.
global_importance_comparison.to_csv(OUTPUT_DIR / 'global_importance_comparison.csv', index=False)
partial_dependence_summary.to_csv(OUTPUT_DIR / 'partial_dependence_summary.csv', index=False)

print('Saved:', OUTPUT_DIR / 'global_importance_comparison.csv')
print('Saved:', OUTPUT_DIR / 'partial_dependence_summary.csv')
global_importance_comparison.sort_values(['dataset', 'model', 'method', 'rank']).groupby(['dataset', 'model', 'method'], group_keys=False).head(5)
